# Seizure codes: how many patients have each code

Seizure types were coded from the pre- and post-op neurology notes against the 2017 ILAE taxonomy (codes 2-38), one code per seizure type the note describes. Every analysis works with **groups** of these codes; this notebook shows the codes themselves: how many patients have each, before and after surgery, and which core group (if any) it counts towards. Improvement by code is in `1_improvement/codes/`, and which codes occur together in `4_combinations/codes/`.

- **Patients:** everyone with pre-op seizure types recorded (100). Post-op counts use everyone with post-op types recorded (95); left / stayed / new use the 95 with both.
- **Core groups**, as in `cc_stats/custom_groups.py` and the curation document's table: atonic (atonic focal, atonic general, myoclonic-atonic), BTC (generalized tonic-clonic, myoclonic-tonic-clonic, focal motor or non-motor to BTC), tonic (general, focal), absence (typical, atypical, myoclonic absence, eyelid myoclonia), myoclonic (general, focal). Codes outside these count only towards the onset groups (general, focal and their combinations). `code_counts.py` checks that each group's codes add up to exactly the `has_<group>` column the analyses use.
- **Left / stayed / new:** among patients with both recorded, a code present before surgery and absent after (left), present at both (stayed), or absent before and present after (new).

`code_counts.py` computes the counts, and this notebook runs it. The output goes to `results/<version>/0_cohort/codes/`.

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch

from cc_stats.paths import DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import GROUP_COLORS, INK, NO_GROUP, pretty, save_figure

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/0_cohort/code_counts.py'],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'code_counts.py failed:\n{run.stderr}')
print(run.stdout.splitlines()[0])
RESULTS = results_dir('0_cohort/codes')
codes = pd.read_csv(RESULTS / 'code_counts.csv', keep_default_na=False)
per_patient = pd.read_csv(RESULTS / 'codes_per_patient.csv', index_col='subject_id')

## Every code

All 37 codes, in code order. Codes 16 and 27 (focal motor / non-motor, unknown awareness) have no column in the workbook, so nobody has them.

In [ ]:
display(pd.DataFrame({
    'Code': codes['code'],
    'Seizure type': codes['label'],
    'Core group': codes['core_group'].replace('', '-'),
    'Pre-op': codes['n_pre'].astype(str) + ' (' + (codes['pct_pre'] * 100).round().astype(int).astype(str) + '%)',
    'Post-op': codes['n_post'],
    'Left': codes['n_left'],
    'Stayed': codes['n_stayed'],
    'New after surgery': codes['n_new_post'],
}).style.hide(axis='index'))

## Pre-op and post-op counts

One row per code anyone has, grouped by onset (generalized, focal, unknown, other). The solid bar is pre-op, the lighter bar post-op; the colour is the core group the code counts towards.

In [ ]:
d = codes[(codes['n_pre'] > 0) | (codes['n_post'] > 0)].reset_index(drop=True)
colors = [GROUP_COLORS.get(g, NO_GROUP) for g in d['core_group']]
y = np.arange(len(d))[::-1]  # code order, top to bottom

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(y + 0.2, d['n_pre'], height=0.4, color=colors)
ax.barh(y - 0.2, d['n_post'], height=0.4, color=colors, alpha=0.4)
for yi, pre, post in zip(y, d['n_pre'], d['n_post']):
    ax.text(pre + 0.5, yi + 0.2, str(pre), va='center', fontsize=8.5, color=INK)
    ax.text(post + 0.5, yi - 0.2, str(post), va='center', fontsize=8.5, color='#888888')
ax.set_yticks(y, d['code'].astype(str) + '  ' + d['label'])
# A line between onset sections
for i in np.flatnonzero(d['onset'].ne(d['onset'].shift()).to_numpy())[1:]:
    ax.axhline(y[i] + 0.5, color='#cccccc', lw=1)
ax.set_xlabel('Patients')
ax.set_xlim(0, d[['n_pre', 'n_post']].to_numpy().max() * 1.1)
ax.set_ylim(-0.6, len(d) - 0.4)
ax.set_title('Patients with each seizure code, before and after surgery', loc='left')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(handles=[Patch(color=c, label=pretty(g)) for g, c in GROUP_COLORS.items()] + [Patch(color=NO_GROUP, label='no core group'),
                   Patch(facecolor='white', edgecolor=INK, label='solid = pre-op'),
                   Patch(facecolor='#dddddd', edgecolor='#dddddd', label='light = post-op')],
          loc='upper center', bbox_to_anchor=(0.4, -0.04), ncol=4, frameon=False)
plt.tight_layout()
save_figure('0_cohort/codes', 'code_counts_pre_post')
plt.show()

## Codes per patient

How many distinct codes each patient had before surgery.

In [ ]:
counts = per_patient['n_codes'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(counts.index, counts.values, color='#3b6fb6')
ax.bar_label(bars)
ax.set_xticks(counts.index)
ax.set_xlabel('Distinct seizure codes before surgery')
ax.set_ylabel('Patients')
ax.set_title(f"Codes per patient (n={len(per_patient)}, median {per_patient['n_codes'].median():.0f})", loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_figure('0_cohort/codes', 'codes_per_patient')
plt.show()

## LGS patients with no tonic code

Tonic seizures are a required feature of LGS under current ILAE criteria, but some patients with the LGS flag have no tonic code (generalized or focal) before surgery. Counts only; no patient IDs are written here.

In [ ]:
from cc_stats.paths import ANALYSIS_CSV

analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
lgs = analysis[(analysis['lgs'] == 1) & analysis['has_tonic'].notna()]
no_tonic = lgs[lgs['has_tonic'] == 0]
print(f"{len(lgs)} LGS patients with pre-op types recorded; {len(no_tonic)} have no tonic code "
      f"({int(no_tonic['engel_improved'].sum())}/{int(no_tonic['engel_improved'].notna().sum())} improved on Engel)")

**Checked 2026-10-20 on v2: 12 of the 47 LGS patients have no tonic code.** Their codes fall into five patterns (a patient can show more than one):

- **Tonic appears only after surgery (3):** focal motor tonic is coded post-op but not pre-op, so tonic seizures were probably present but not described in the pre-op note.
- **Drop-attack profile without tonic (5):** generalized atonic and / or tonic-clonic, with myoclonic or atypical absence in some. Otherwise an LGS-like profile; drop attacks described as "atonic" may include tonic drops.
- **Mostly or only focal codes (3):** focal motor or non-motor seizures with focal to BTC; two patients have focal codes only. Unusual for LGS; worth confirming the diagnosis.
- **Mostly unknown-onset codes (2):** unknown motor tonic-clonic alone in one; unknown-onset codes with generalized atonic in the other, suggesting thin notes.
- **Evolving from infantile spasms (1):** age 2, epileptic spasms with myoclonic and atonic seizures; tonic seizures often appear later in this transition.

One further patient has only tonic-clonic and *typical* absence seizures, which also sits poorly with LGS (its absences are usually atypical). So for roughly 9 of the 12 the pattern is consistent with LGS with tonic seizures under-documented or not yet present, and for about 4 the diagnosis is worth checking with the clinical team. The myoclonic-within-LGS comparison is re-run on LGS patients with a tonic code in `4_combinations/types/combinations_and_outcome.ipynb` (and its codes counterpart).